In [ ]:
import geopandas as gpd
import pandas as pd
import numpy as np
import xarray as xr
import os
import matplotlib.pyplot as plt
from pyproj import CRS, Transformer
import matplotlib.lines as mlines
import regionmask
import multiprocessing as mp

## Create masks

In [ ]:
ba = xr.open_mfdataset('/net/rain/hyclimm/data/public/hazard/obs/fire_observations/FireCCI/pixel/v511_2000-2019_Modis/Modis_BA_pixel_2020/20200801-ESACCI-L3S_FIRE-BA-MODIS-AREA_3-fv5.1.1cds.nc',
                       preprocess = lambda ds: ds.sel(lon = slice(-6, 10), lat = slice(52, 41)))
# PRUDENCE France
reg10 = gpd.read_file('/net/rain/hyclimm/data/projects/SynFire/WP1/Regionalization/Study_Area_32E_10_regions.shp')
fr_p = reg10.loc[reg10['region'] == 'France']

# France
study_area = gpd.read_file('/net/rain/hyclimm/data/projects/SynFire/Study_Area/Study_Area_32E.shp')
fr = study_area.loc[study_area['CNTR_NAME'] == 'France']

fr_p_mask = regionmask.mask_geopandas(fr_p, ba.lon.values, ba.lat.values)
fr_mask = regionmask.mask_geopandas(fr, ba.lon.values, ba.lat.values)

In [ ]:
#convert to dataset
fr_p_mask = fr_p_mask.to_dataset()
fr_mask = fr_mask.to_dataset()

In [ ]:
#save
fr_mask.to_netcdf('/net/rain/hyclimm/data/projects/SynFire/WP1/France_fire_seasonality/France_mask.nc')
fr_p_mask.to_netcdf('/net/rain/hyclimm/data/projects/SynFire/WP1/France_fire_seasonality/PRUDENCE_France_mask.nc')

## calculate fire seasonality of France and PRUDENCE France (FRY)

In [ ]:
fire_obs = gpd.read_file('/net/rain/hyclimm/data/projects/SynFire/WP1/Regionalization/FRYv2.0_FireCCI51_6D_2001-2020_study_area_fire_observations_w_region_w_country.shp')

In [ ]:
fire_seas = pd.DataFrame(columns = ["Region", "Month", "EV", "BA"]) #region, month, number of fire event, cumulative burned area
months = ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]

In [ ]:
for reg in ['FR', 'FRA']:

    if reg == 'FR':
        fire_obs_reg = fire_obs[fire_obs["region"] == reg].copy()
    else:
        fire_obs_reg = fire_obs[fire_obs["country"] == reg].copy()
    
    fire_obs_reg["start_date"] = pd.to_datetime(fire_obs_reg["start_date"])
    fire_seas_reg = pd.DataFrame(columns = ["Region", "Month", "EV", "BA"])
    fire_seas_reg["Region"] = [reg for _ in range(12)]
    fire_seas_reg["Month"] = months
    
    for ind, mon in enumerate(months):
        
        fire_obs_reg_mon = fire_obs_reg[fire_obs_reg['start_date'].dt.month == ind+1]
        
        #avg number of events across the 20 yrs period
        EV = len(fire_obs_reg_mon)/20
        
        #avg cumulative burned area across the 20 yrs period
        BA = fire_obs_reg_mon["area"].sum()/20

        fire_seas_reg.loc[fire_seas_reg["Month"] == mon, "EV"] = EV
        fire_seas_reg.loc[fire_seas_reg["Month"] == mon, "BA"] = BA

    # rescaled to monthly maximal values
    fire_seas_reg["EV"] = fire_seas_reg["EV"]/(fire_seas_reg["EV"].max())
    fire_seas_reg["BA"] = fire_seas_reg["BA"]/(fire_seas_reg["BA"].max())

    fire_seas = pd.concat([fire_seas, fire_seas_reg], ignore_index = True)

## plot

In [ ]:
fig = plt.figure(figsize=(15, 5))

ax1 = fig.add_subplot(1, 3, 1)  # normal
ax2 = fig.add_subplot(1, 3, 2, projection = 'polar')  # normal
ax3 = fig.add_subplot(1, 3, 3, projection = 'polar')  # polar

#1.plot regions
# PRUDENCE France
reg10_box = gpd.read_file('/net/rain/hyclimm/data/projects/SynFire/WP1/Regionalization/Region_Ten.shp')
fr_p = reg10_box.loc[reg10_box['region'] == 'France']

# France
study_area = gpd.read_file('/net/rain/hyclimm/data/projects/SynFire/Study_Area/Study_Area_32E.shp')
fr = study_area.loc[study_area['CNTR_NAME'] == 'France']

fr_p.boundary.plot(ax = ax1, color = 'red', label = 'PRUDENCE FR', linewidth = 2)
fr.boundary.plot(ax = ax1, color = 'black', label = 'France', linewidth = 2)

ax1.legend(frameon = False, loc = 'lower left', fontsize = 12, bbox_to_anchor = (0, -0.04))
ax1.set_xlabel('lon', fontsize = 15)
ax1.set_ylabel('lat', fontsize = 15)

#2. seasonality PRUDENCE FR and France
for (ax, reg) in zip([ax2, ax3], ["FR", "FRA"]):
    
    #get data and angles
    fire_seas_reg = fire_seas[fire_seas["Region"] == reg]
    num_ev = list(fire_seas_reg["EV"])
    burned_area = list(fire_seas_reg["BA"])
    angles = np.linspace(0, 2 * np.pi, len(months), endpoint=False).tolist()
    
    # close the loop, append the first to the end of the list
    num_ev += num_ev[:1]
    burned_area += burned_area[:1]
    angles += angles[:1]
    
    # Draw one axe per variable and add labels
    ax.set_theta_offset(np.pi / 2)
    ax.set_theta_direction(-1)
    ax.set_xticks(angles[:-1], months, fontsize = 12)
    ax.tick_params(axis='x', pad=8)
    
    # Draw ylabels
    ax.set_rlabel_position(0)
    ax.set_yticks([0, 0.2, 0.4, 0.6, 0.8, 1.0], ["0", "0.2", "0.4", "0.6", "0.8", "1.0"], color="black", fontsize=12)
    ax.set_ylim(0, 1)
    
    # Plot data
    ax.plot(angles, burned_area, color='#33A036', linewidth=2, linestyle='solid', label='BA')
    ax.fill(angles, burned_area, color='#33A036', alpha=0.25)
    
    ax.plot(angles, num_ev, color='blue', linewidth=2, linestyle='solid', label='FO')
    ax.fill(angles, num_ev, color='blue', alpha=0.25)
    
    ax.set_title(reg, fontsize = 18)

ax1.text(0.05, 0.9, '(a)', fontsize = 16, transform = ax1.transAxes)
ax2.set_title('(b) PRUDENCE FR', fontsize = 16)
ax3.set_title('(c) France', fontsize = 16)

#legend
blue_square = mlines.Line2D([], [], marker = "s", markersize = 16, markerfacecolor = (0, 0, 1, 0.25), 
                            markeredgecolor = "blue", markeredgewidth = 2, linestyle = "None", 
                            label = "Fire event number")
green_square = mlines.Line2D([], [], marker = "s", markersize = 16, markerfacecolor =  ('#33A036', 0.25), 
                             markeredgecolor = "#33A036", markeredgewidth = 2, linestyle = "None", 
                             label = "Burned area")
ax2.legend(handles = [blue_square, green_square], 
           loc='upper left', fontsize=12, title='', frameon = False, ncol = 2, 
           bbox_to_anchor=(0.55, -0.1), handletextpad = 0, labelspacing = 1)

plt.subplots_adjust(wspace = 0.3)

plt.savefig('/home/lixinh/SynFire_Scripts/WP1/Figures/R_CEE_France_fire_seasonality.png', dpi = 600, bbox_inches = 'tight')